## 大项目开始了！！

# "THE PRICE IS RIGHT" 顶点项目

本周——基于抓取的 Amazon 数据，构建一个能根据描述预测某物价格的模型

# 日程安排

DAY 1：数据整理（Data Curation）  
DAY 2：数据预处理（Data Pre-processing）  
DAY 3：评估、基线、传统机器学习  
DAY 4：深度学习与 LLM  
DAY 5：微调前沿模型  

## DAY 1：数据整理

今天我们将清洗数据集并整理数据

数据集在这里：  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

包含所有产品数据集的文件夹在这里：  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">数据整理的商业价值</h2>
            <span style="color:#181;">数据整理常被视为数据科学家工作中不那么光鲜的部分。我说那是胡说！
            科学正是发生在这里——还有什么比这更光鲜的呢？！对数据集做研发，往往比我们稍后做的时髦的「超参数优化」对性能影响更大。
            所以：准备好与数据质量共度高质量时光吧。</span>
        </td>
    </tr>
</table>

In [ ]:
# 导入

import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from pricer.items import Item
from pricer.parser import parse
load_dotenv(override=True)

In [ ]:
# 登录 Hugging Face——如果看到关于 Environment variable 已设置的 “Note”，忽略即可

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

## 加载我们的数据集

在下一个单元格中，我们从 Hugging Face 加载数据集。

如果出现类似 “trust_remote_code is no longer supported” 的错误，请在新单元格中运行：`!uv add --upgrade datasets==3.6.0`，然后重启 Kernel，再试一次。

In [ ]:
# 从 Hugging Face Hub 加载 Amazon Reviews 2023 家电类原始元数据

dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)

In [ ]:
# 打印该子集有多少条商品

print(f"Number of Appliances: {len(dataset):,}")

In [ ]:
# 查看某一个数据点

dataset[6]


In [ ]:

# 最贵的商品是什么？

max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

这是我能找到的最接近的——看起来像是超值价！！

https://www.amazon.com/TurboChef-Electric-Countertop-Microwave-Convection/dp/B01D05U9NO/

In [ ]:
# 若价格在 $1–$1000 且细节足够，则加载为 Item 对象

items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
# 查看解析后的第一个 Item 对象

items[0]

In [ ]:
# full：把标题、描述等拼成的完整文本，后面做特征/微调都会用到

print(items[0].full)

In [ ]:
# 抽出价格与文本长度，准备画分布图

prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

In [ ]:
# 绘制长度分布：看商品描述有多长（影响上下文与费用）

plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
# 找到文本最长的商品并打印

max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)


In [ ]:
# 绘制价格分布
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# 再看一条样本的 full 文本

print(items[3].full)

In [ ]:
# 用课程封装的 ItemLoader 重新加载（含清洗逻辑）

from pricer.loaders import ItemLoader
loader = ItemLoader("Appliances")
items = loader.load()


In [ ]:
# 多个亚马逊品类名称列表，后面会逐个加载

dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

In [ ]:
# 遍历所有品类，合并到一个大 items 列表

items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())

In [ ]:
# 合计商品数量

print(f"A grand total of {len(items):,} items")

In [ ]:
# 抽查第 1000 个样本

items[1000]

In [ ]:
# 洗牌后去重：先按 title，再按 full 文本，避免重复样本污染训练

random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
# 去重后再次查看文本长度分布

lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
# 绘制价格分布

prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# 各类别样本量柱状图——看类别是否严重不平衡

from collections import Counter
category_counts = Counter([item.category for item in items])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

In [ ]:
# 加权抽样：提高高价样本权重，并下调 Automotive 等过量类别
# 得到约 SIZE 条更均衡、更适合学定价的数据

np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]


In [ ]:
# 抽样后价格分布

prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# 为保险起见，对最终数据集再洗牌一次样本

random.seed(42)
random.shuffle(sample)


In [ ]:
# 最终洗牌后的价格分布确认

prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in sample])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

# 按类别的柱状图
plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')

plt.xticks(rotation=30, ha='right')

# 在每根柱子顶部添加数值标签
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

# 显示图表
plt.show()

In [ ]:
# Automotive 仍领先，但已有所改善
# 换个视角，看看饼图

plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)

# 在中心加一个圆，做成环形图（可选）
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Categories')

# 相等的纵横比确保饼图画成正圆
plt.axis('equal')  

plt.show()

In [ ]:
# 价格如何随字符数变化？

sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

# 创建散点图
plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")

# 添加标签和标题
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')

# 显示图
plt.show()

In [ ]:
# 价格如何随重量变化？

ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

# 创建散点图
plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")

# 添加标签和标题
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')

# 显示图
plt.show()

## 现在把这个数据集推送到 Hugging Face Hub

如果你自己构建了数据集，请把用户名替换成你的 HF 用户名

或者忽略这个单元格，明天你可以加载我的数据集！

In [ ]:
# 划分 train / val / test，并推送到 Hugging Face Hub
# 同时上传 full 与 lite（小）两套，方便后面快速实验

username = "ed-donner"
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## 旁注

如果你喜欢 matplotlib 图表里可用的各种颜色，应该收藏这个：

https://matplotlib.org/stable/gallery/color/named_colors.html
